# From raw ERA5 to credible wind capacity factors: a worked example

This notebook reproduces, end to end and on **only open data**, the core result of the
PyVWF bias correction: turning raw ERA5 reanalysis wind into validated wind capacity
factors, and measuring how much the correction improves them against real observations.

The region is **Denmark, onshore**: 4,866 turbines metered by the Danish Energy Agency,
trained on 2015-2019 and validated on the held-out year 2020. Every turbine is given a
real power curve and hub height matched from the **bundled open curve library** (69 real
machines from the NREL/turbine-models archive), so nothing here needs a licensed dataset
and a stranger can run it as-is.

**Headline result:** the correction roughly halves the held-out capacity-factor RMSE
(0.158 to 0.085) and removes a +12 percentage-point systematic bias, and the skill
saturates around 200 spatial clusters. Time resolution (monthly vs seasonal vs annual)
barely matters.

![DK onshore RMSE vs cluster count](../docs/findings/figures/dk_onshore_rmse_vs_k.png)

*Held-out capacity-factor RMSE against the number of spatial clusters, for four time
resolutions. The dashed line is uncorrected ERA5. Full sweep and discussion:
`docs/findings/method-cluster-count-dk.md`.*

## Setup

Run from the repo root with the combined input tree pointed at by `PYVWF_INPUT`:

```bash
PYVWF_INPUT=input/combined PYTHONPATH=src jupyter lab examples/dk_raw_vs_corrected.ipynb
```

The correction is embarrassingly parallel in its offset fit; set
`PYVWF_OFFSET_WORKERS=4` to fan it across cores for larger cluster counts.

In [ ]:
import os, warnings, dataclasses
from pathlib import Path
os.environ.setdefault('PYVWF_INPUT', 'input/combined')
warnings.simplefilter('ignore')

import pandas as pd
from vwf.harness.regions import load_region
from vwf.harness.driver import run_train, run_evaluate

spec = load_region(Path('configs/regions/dk.toml'))
print(spec.name, '| train', spec.train_years, '-> test', spec.test_years)

## Train and evaluate one configuration

We use a modest 50 clusters and a single annual (`fixed`) correction so the notebook runs
in a couple of minutes; the full sweep in the findings doc goes to 3,300 clusters. The
harness writes the fitted factors and a `metrics.csv` with the uncorrected baseline plus
each corrected variant.

In [ ]:
spec50 = dataclasses.replace(spec, cluster_list=(1, 50), time_slices=('fixed', 'season'))
out = Path('output/examples/dk_raw_vs_corrected')

train_dir = run_train(spec50, out, run_name='demo')
eval_dir = run_evaluate(spec50, train_dir, out, run_name='demo')

metrics = pd.read_csv(Path(eval_dir) / 'metrics.csv')
metrics[['variant', 'num_clu', 'time_res', 'rmse', 'mbe', 'pearson_r']]

In [ ]:
unc = metrics[metrics.variant == 'uncorrected'].iloc[0]
best = metrics[metrics.variant == 'affine-wind'].sort_values('rmse').iloc[0]
print(f"uncorrected : RMSE {unc.rmse:.3f}  bias {unc.mbe:+.3f}  r {unc.pearson_r:.3f}")
print(f"corrected   : RMSE {best.rmse:.3f}  bias {best.mbe:+.3f}  r {best.pearson_r:.3f}"
      f"   (k={int(best.num_clu)}, {best.time_res})")
print(f"RMSE reduction: {100*(1 - best.rmse/unc.rmse):.0f}%")

## The full cluster sweep

The figure at the top comes from sweeping the cluster count across the paper's grid. If you
have run that sweep (`docs/findings/method-cluster-count-dk.md`), its combined
metrics reproduce the plot directly:

In [ ]:
import matplotlib.pyplot as plt

sweep_path = Path('output/validation/dk_onshore_sweep_2026-07-24/combined_metrics.csv')
if sweep_path.exists():
    s = pd.read_csv(sweep_path)
    corr = s[s.variant == 'affine-wind']
    u = s[s.variant == 'uncorrected'].iloc[0]
    piv = corr.pivot_table(index='num_clu', columns='time_res', values='rmse')
    ax = piv[['month', 'bimonth', 'season', 'fixed']].plot(marker='o', ms=4, logx=True, figsize=(7, 4.4))
    ax.axhline(u.rmse, ls='--', color='crimson', label=f'uncorrected ({u.rmse:.3f})')
    ax.set(xlabel='spatial clusters (k)', ylabel='held-out capacity-factor RMSE')
    ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3)
else:
    print('Run the sweep first (scripts referenced in the findings doc) to reproduce the figure.')

## What this shows, and how to point it at your own fleet

- **Correction, not just rescaling.** RMSE falls from 0.158 to ~0.085 and the systematic
  bias from +0.12 to +0.02: ERA5's Danish wind bias is close to a clusterable wind-speed
  offset, which is exactly what the affine correction represents.
- **Cluster count is the lever, and it saturates.** Skill floors around k=200; going to
  3,300 buys almost nothing. Time resolution barely matters.
- **Real curves matter.** Every turbine here uses a matched real power curve and hub
  height, not a uniform default; a uniform curve lets the 'correction' launder a curve
  error that then fails to generalise.

To run this on **your** fleet, implement a small `ObservationSource` adapter that yields
your turbine metadata and monthly generation, point a region config at it, and the same
`run_train` / `run_evaluate` pipeline produces your validated correction. See the harness
docs and `src/vwf/sources/` for the adapter contract.

**Scope note:** this is screening-level validation on one held-out year, not an accredited
yield assessment.